# Importing Requirements

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

if not os.path.exists('../plots'):
    os.makedirs('../plots')
if not os.path.exists('../report/tables'):
    os.makedirs('../report/tables')


# Data Input

In [25]:
# constants and data
d = 430  # distance to screen in mm
g = 1671  # grating constant in nm (1.671 um)
central_max = 560  # mm

# Measured positions on ruler (mm)
pos_red = 740     # turuncu 
pos_green = 730   # yeşil 
pos_blue = 700    # mavi 

# Calculate 2l
# absolute position on the ruler. 
# l is the distance from the central maximum.
l_red = abs(pos_red - central_max)
l_green = abs(pos_green - central_max)
l_blue = abs(pos_blue - central_max)

print(f"l_red: {l_red} mm")
print(f"l_green: {l_green} mm")
print(f"l_blue: {l_blue} mm")


l_red: 180 mm
l_green: 170 mm
l_blue: 140 mm


# Calculations

In [26]:
# Calculate experimental wavelengths
def calc_wavelength(l, d, g):
    return g * l / np.sqrt(d**2 + l**2)

lambda_exp_red = calc_wavelength(l_red, d, g)
lambda_exp_green = calc_wavelength(l_green, d, g)
lambda_exp_blue = calc_wavelength(l_blue, d, g)

print(f"Experimental Wavelength (Red/H_alpha): {lambda_exp_red:.2f} nm")
print(f"Experimental Wavelength (Green/H_beta): {lambda_exp_green:.2f} nm")
print(f"Experimental Wavelength (Blue/H_gamma): {lambda_exp_blue:.2f} nm")


Experimental Wavelength (Red/H_alpha): 645.24 nm
Experimental Wavelength (Green/H_beta): 614.36 nm
Experimental Wavelength (Blue/H_gamma): 517.32 nm


In [27]:
# table 1 data preparation
# for Hydrogen lines:

data = {
    'Line': [r'$H_{\alpha}$', r'$H_{\beta}$', '$H_{\gamma}$'],
    '2l (m)': [2 * l_red / 1000, 2 * l_green / 1000, 2 * l_blue / 1000],  # 2l in meters
    'n1': [2, 2, 2],
    'n2': [3, 4, 5],
    '$\lambda_{exp}$ (nm)': [lambda_exp_red, lambda_exp_green, lambda_exp_blue],
    '$\lambda_{lit}$ (nm)': [656.28, 486.13, 434.05]
}

df1 = pd.DataFrame(data)

# rydberg constant
# 1/lambda = R_th * (1/n1^2 - 1/n2^2)
# R_exp = 1 / (lambda_exp * (1/n1^2 - 1/n2^2))
# R_exp in m^-1
df1['$R_{exp}$ (m^-1)'] = 1 / ((df1['$\lambda_{exp}$ (nm)'] * 1e-9) * (1/(df1['n1']**2) - 1/(df1['n2']**2)))

R_ave = df1['$R_{exp}$ (m^-1)'].mean()
R_std = df1['$R_{exp}$ (m^-1)'].std()

print(df1)
print(f"\nAverage Rydberg Constant: {R_ave:.2e} m^-1 +/- {R_std:.2e} m^-1")

df1_export = df1.copy()
df1_export['$\lambda_{exp}$ (nm)'] = df1_export['$\lambda_{exp}$ (nm)'].map('{:.2f}'.format)
df1_export['$R_{exp}$ (m^-1)'] = df1_export['$R_{exp}$ (m^-1)'].map('{:.3e}'.format)

df1_export.columns = ['Line', '2l', 'n1', 'n2', 'lambdaExp', 'lambdaLit', 'Rexp']
df1_export.to_csv('../report/tables/table1.csv', index=False)


           Line  2l (m)  n1  n2  $\lambda_{exp}$ (nm)  $\lambda_{lit}$ (nm)  \
0  $H_{\alpha}$    0.36   2   3            645.236745                656.28   
1   $H_{\beta}$    0.34   2   4            614.358004                486.13   
2  $H_{\gamma}$    0.28   2   5            517.318285                434.05   

   $R_{exp}$ (m^-1)  
0      1.115869e+07  
1      8.681149e+06  
2      9.204981e+06  

Average Rydberg Constant: 9.68e+06 m^-1 +/- 1.31e+06 m^-1


# Plots

In [28]:
# plot
plt.figure(figsize=(8, 6))
plt.scatter(df1['\lambda_{lit} (nm)'], df1['\lambda_{exp} (nm)'], color='blue', label='Data Points', s=100)

min_val = min(df1['lambda_lit (nm)'].min(), df1['lambda_exp (nm)'].min()) - 20
max_val = max(df1['lambda_lit (nm)'].max(), df1['lambda_exp (nm)'].max()) + 20
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal Match')

plt.title('Experimental vs Literature Wavelengths for Hydrogen')
plt.xlabel('Literature Wavelength (nm)')
plt.ylabel('Experimental Wavelength (nm)')
plt.grid(True)
plt.legend()
plt.savefig('../plots/wavelength_comparison.png', dpi=300)
plt.show()


KeyError: '\\lambda_{lit} (nm)'

<Figure size 800x600 with 0 Axes>